In [1]:
import pandas as pd

# 1. Filtrando ruas_poa.csv
# Colunas para manter: osmid, name, highway, length, lanes, maxspeed, geometry
ruas_cols = ['osmid', 'name', 'highway', 'length', 'lanes', 'maxspeed', 'geometry']
try:
    df_ruas = pd.read_csv('/content/ruas_poa.csv')
    # Manter apenas as colunas que realmente existem no dataset de interesse
    cols_to_keep_ruas = [col for col in ruas_cols if col in df_ruas.columns]
    df_ruas_filtered = df_ruas[cols_to_keep_ruas]
    df_ruas_filtered.to_csv('/content/ruas_poa_filtered.csv', index=False)
    print("ruas_poa.csv filtrado com sucesso!")
    display(df_ruas_filtered.head(2))
except Exception as e:
    print(f"Erro ao processar ruas_poa.csv: {e}")

ruas_poa.csv filtrado com sucesso!


,osmid,name,highway,length,lanes,maxspeed,geometry
0,"[447936235, 447936237]",Freeway,motorway,318.966066,"['5', '6']",100,"LINESTRING (-51.2007466 -29.979128, -51.201557..."
1,29474897,Freeway,motorway,238.530723,4,100,"LINESTRING (-51.1793595 -29.9698178, -51.18182..."


In [2]:
# 2. Filtrando pois_poa.csv
# Colunas para manter: name, amenity, public_transport, geometry
pois_cols = ['name', 'amenity', 'public_transport', 'geometry']
try:
    df_pois = pd.read_csv('/content/pois_poa.csv')
    cols_to_keep_pois = [col for col in pois_cols if col in df_pois.columns]
    df_pois_filtered = df_pois[cols_to_keep_pois]
    df_pois_filtered.to_csv('/content/pois_poa_filtered.csv', index=False)
    print("\npois_poa.csv filtrado com sucesso!")
    display(df_pois_filtered.head(2))
except Exception as e:
    print(f"Erro ao processar pois_poa.csv: {e}")


pois_poa.csv filtrado com sucesso!


,name,amenity,public_transport,geometry
0,NaN,NaN,stop_position,POINT (-51.2197338 -30.0233716)
1,NaN,NaN,stop_position,POINT (-51.1881767 -30.0318426)


In [3]:
# 3. Filtrando areas_verdes_poa.csv
# Colunas para manter: name, leisure, landuse, geometry
verdes_cols = ['name', 'leisure', 'landuse', 'geometry']
try:
    df_verdes = pd.read_csv('/content/areas_verdes_poa.csv')
    cols_to_keep_verdes = [col for col in verdes_cols if col in df_verdes.columns]
    df_verdes_filtered = df_verdes[cols_to_keep_verdes]
    df_verdes_filtered.to_csv('/content/areas_verdes_poa_filtered.csv', index=False)
    print("\nareas_verdes_poa.csv filtrado com sucesso!")
    display(df_verdes_filtered.head(2))
except Exception as e:
    print(f"Erro ao processar areas_verdes_poa.csv: {e}")


areas_verdes_poa.csv filtrado com sucesso!


,name,leisure,landuse,geometry
0,NaN,garden,NaN,POINT (-51.2542792 -30.1048299)
1,Jardim Comunitário,garden,NaN,POINT (-51.144836 -30.0176925)


In [5]:
# 4. Filtrando cat_acidentes.csv
# Colunas para manter: data, tipo_acid, feridos, mortes, bicicleta, latitude, longitude
# Filtrar linhas: remover latitude ou longitude vazia
acidentes_cols = ['data', 'tipo_acid', 'feridos', 'mortes', 'bicicleta', 'latitude', 'longitude']
try:
    # Carregar especificando o separador ';' já que os dados estão separados dessa forma
    df_acidentes = pd.read_csv('/content/cat_acidentes.csv', sep=';')
    cols_to_keep_acid = [col for col in acidentes_cols if col in df_acidentes.columns]
    df_acid_filtered = df_acidentes[cols_to_keep_acid].copy()

    # Remover linhas onde latitude ou longitude estão vazias
    if 'latitude' in df_acid_filtered.columns and 'longitude' in df_acid_filtered.columns:
        df_acid_filtered = df_acid_filtered.dropna(subset=['latitude', 'longitude'])

    df_acid_filtered.to_csv('/content/cat_acidentes_filtered.csv', index=False)
    print("\ncat_acidentes.csv filtrado com sucesso!")
    display(df_acid_filtered.head(2))
except Exception as e:
    print(f"Erro ao processar cat_acidentes.csv: {e}")


cat_acidentes.csv filtrado com sucesso!


,data,tipo_acid,feridos,mortes,bicicleta,latitude,longitude
0,2021-01-02 00:00:00,ABALROAMENTO,6,0,0,-30.014936,-51.119179
1,2021-01-03 00:00:00,CHOQUE,1,0,0,-30.162425,-51.157411


In [6]:
# Mostrando a quantidade real de registros e colunas em cada dataset filtrado
print(f"Quantidade de linhas em ruas_poa_filtered.csv: {df_ruas_filtered.shape[0]:,}".replace(',', '.'))
print(f"Quantidade de linhas em pois_poa_filtered.csv: {df_pois_filtered.shape[0]:,}".replace(',', '.'))
print(f"Quantidade de linhas em areas_verdes_poa_filtered.csv: {df_verdes_filtered.shape[0]:,}".replace(',', '.'))
print(f"Quantidade de linhas em cat_acidentes_filtered.csv: {df_acid_filtered.shape[0]:,}".replace(',', '.'))

Quantidade de linhas em ruas_poa_filtered.csv: 50.410
Quantidade de linhas em pois_poa_filtered.csv: 691
Quantidade de linhas em areas_verdes_poa_filtered.csv: 1.679
Quantidade de linhas em cat_acidentes_filtered.csv: 80.179


In [7]:
!pip install geopandas shapely -q

### Criando os GeoDataFrames a partir das geometrias textuais

Vamos ler os arquivos filtrados e converter a coluna `geometry` (que está salva como texto) em objetos geométricos reais utilizando a biblioteca `shapely` e o `geopandas`.

In [8]:
import geopandas as gpd
from shapely import wkt

# Carregar os arquivos filtrados
df_ruas_raw = pd.read_csv('/content/ruas_poa_filtered.csv')
df_pois_raw = pd.read_csv('/content/pois_poa_filtered.csv')
df_verdes_raw = pd.read_csv('/content/areas_verdes_poa_filtered.csv')

# Tratar possíveis valores nulos na geometria antes de converter
df_ruas_raw = df_ruas_raw.dropna(subset=['geometry'])
df_pois_raw = df_pois_raw.dropna(subset=['geometry'])
df_verdes_raw = df_verdes_raw.dropna(subset=['geometry'])

# Converter a coluna de texto para geometria real do Shapely
df_ruas_raw['geometry'] = df_ruas_raw['geometry'].apply(wkt.loads)
df_pois_raw['geometry'] = df_pois_raw['geometry'].apply(wkt.loads)
df_verdes_raw['geometry'] = df_verdes_raw['geometry'].apply(wkt.loads)

# Criar os GeoDataFrames definindo o CRS inicial como WGS84 (EPSG:4326)
gdf_ruas = gpd.GeoDataFrame(df_ruas_raw, geometry='geometry', crs='EPSG:4326')
gdf_pois = gpd.GeoDataFrame(df_pois_raw, geometry='geometry', crs='EPSG:4326')
gdf_verdes = gpd.GeoDataFrame(df_verdes_raw, geometry='geometry', crs='EPSG:4326')

print("Conversão para GeoDataFrames realizada com sucesso!")
print(f"Ruas: {gdf_ruas.shape[0]} | POIs: {gdf_pois.shape[0]} | Áreas Verdes: {gdf_verdes.shape[0]}")

Conversão para GeoDataFrames realizada com sucesso!
Ruas: 50410 | POIs: 690 | Áreas Verdes: 1679


### Cruzamento Espacial (Exemplo: Contar POIs próximos de cada Rua)

Para que sua ML use isso como variável, podemos projetar os dados para um sistema métrico (como o **UTM 22S - EPSG:32722** adequado para Porto Alegre) e criar uma zona de influência (buffer) ao redor de cada rua para contar quantos destinos (POIs) estão nela.

In [9]:
# 1. Mudar para coordenadas métricas (metros) para fazer medições precisas de distância
gdf_ruas_m = gdf_ruas.to_crs(epsg=32722)
gdf_pois_m = gdf_pois.to_crs(epsg=32722)

# 2. Criar um buffer (área de busca) de, por exemplo, 50 metros ao redor do eixo de cada rua
gdf_ruas_buffer = gdf_ruas_m.copy()
gdf_ruas_buffer['geometry'] = gdf_ruas_m.geometry.buffer(50) # 50 metros

# 3. Fazer um Join Espacial (sjoin) entre as áreas de busca das ruas e os POIs
# Isso vai correlcionar cada POI à respectiva rua em que ele está próximo
join_ruas_pois = gpd.sjoin(gdf_pois_m, gdf_ruas_buffer, how='inner', predicate='within')

# 4. Agrupar os dados para contar quantos POIs existem em cada rua (pelo osmid)
contagem_pois_por_rua = join_ruas_pois.groupby('osmid').size().reset_index(name='qtd_pois')

# 5. Unir essa nova coluna (variável explicativa para ML!) de volta no nosso dataset de ruas original
gdf_ruas_final = gdf_ruas.merge(contagem_pois_por_rua, on='osmid', how='left')
gdf_ruas_final['qtd_pois'] = gdf_ruas_final['qtd_pois'].fillna(0).astype(int)

print("Variável 'qtd_pois' integrada com sucesso ao dataset de ruas!")
display(gdf_ruas_final[['osmid', 'name', 'qtd_pois']].head(10))

Variável 'qtd_pois' integrada com sucesso ao dataset de ruas!


,osmid,name,qtd_pois
0,"[447936235, 447936237]",Freeway,0
1,29474897,Freeway,0
2,"[27090408, 1187268413]",NaN,0
3,"[29210971, 288127021, 29210975, 1282045357]",Freeway,0
4,306683729,Rua Albion,0
5,451653212,Rua Albion,0
6,208803356,Rua Albion,0
7,"[451652985, 927886438]",Avenida Ipiranga,0
8,"[356396032, 471814874, 927931452, 471814872]",Avenida Joaquim Porto Villanova,0
9,927886434,Avenida Ipiranga,0


In [10]:
# 1. Mudar o sistema de coordenadas das áreas verdes para metros
gdf_verdes_m = gdf_verdes.to_crs(epsg=32722)

# 2. Fazer o Join Espacial (sjoin) entre a área de busca das ruas (buffer de 50m já criado) e as áreas verdes
join_ruas_verdes = gpd.sjoin(gdf_verdes_m, gdf_ruas_buffer, how='inner', predicate='within')

# 3. Agrupar os dados para contar quantas áreas verdes existem próximas a cada rua (pelo osmid)
contagem_verdes_por_rua = join_ruas_verdes.groupby('osmid').size().reset_index(name='qtd_areas_verdes')

# 4. Unir essa nova coluna ao nosso dataset 'gdf_ruas_final' que já possuía a contagem de POIs
gdf_ruas_final = gdf_ruas_final.merge(contagem_verdes_por_rua, on='osmid', how='left')
gdf_ruas_final['qtd_areas_verdes'] = gdf_ruas_final['qtd_areas_verdes'].fillna(0).astype(int)

# Salvar o dataset espacial final consolidado para a sua ML
gdf_ruas_final.to_csv('/content/ruas_com_atributos_espaciais.csv', index=False)

print("Processamento concluído com sucesso!")
print("O arquivo 'ruas_com_atributos_espaciais.csv' foi gerado contendo as novas colunas 'qtd_pois' e 'qtd_areas_verdes'.")
display(gdf_ruas_final[['osmid', 'name', 'qtd_pois', 'qtd_areas_verdes']].head(10))

Processamento concluído com sucesso!
O arquivo 'ruas_com_atributos_espaciais.csv' foi gerado contendo as novas colunas 'qtd_pois' e 'qtd_areas_verdes'.


,osmid,name,qtd_pois,qtd_areas_verdes
0,"[447936235, 447936237]",Freeway,0,0
1,29474897,Freeway,0,0
2,"[27090408, 1187268413]",NaN,0,0
3,"[29210971, 288127021, 29210975, 1282045357]",Freeway,0,1
4,306683729,Rua Albion,0,0
5,451653212,Rua Albion,0,0
6,208803356,Rua Albion,0,2
7,"[451652985, 927886438]",Avenida Ipiranga,0,0
8,"[356396032, 471814874, 927931452, 471814872]",Avenida Joaquim Porto Villanova,0,0
9,927886434,Avenida Ipiranga,0,0


In [12]:
# 1. Mudar o sistema de coordenadas das áreas verdes para metros
gdf_verdes_m = gdf_verdes.to_crs(epsg=32722)

# 2. Fazer o Join Espacial (sjoin) entre a área de busca das ruas (buffer de 50m já criado) e as áreas verdes
join_ruas_verdes = gpd.sjoin(gdf_verdes_m, gdf_ruas_buffer, how='inner', predicate='within')

# 3. Agrupar os dados para contar quantas áreas verdes existem próximas a cada rua (pelo osmid)
contagem_verdes_por_rua = join_ruas_verdes.groupby('osmid').size().reset_index(name='qtd_areas_verdes')

# 4. Remover a coluna se ela já existir no dataframe para evitar duplicidade de colunas (e o erro de KeyError)
if 'qtd_areas_verdes' in gdf_ruas_final.columns:
    gdf_ruas_final = gdf_ruas_final.drop(columns=['qtd_areas_verdes'])

# Unir de forma segura
gdf_ruas_final = gdf_ruas_final.merge(contagem_verdes_por_rua, on='osmid', how='left')
gdf_ruas_final['qtd_areas_verdes'] = gdf_ruas_final['qtd_areas_verdes'].fillna(0).astype(int)

# Salvar o dataset espacial final consolidado para a sua ML
gdf_ruas_final.to_csv('/content/ruas_com_atributos_espaciais.csv', index=False)

print("Processamento concluído com sucesso (seguro contra re-execução)!")
print("O arquivo 'ruas_com_atributos_espaciais.csv' foi atualizado.")
display(gdf_ruas_final[['osmid', 'name', 'qtd_pois', 'qtd_areas_verdes']].head(10))

Processamento concluído com sucesso (seguro contra re-execução)!
O arquivo 'ruas_com_atributos_espaciais.csv' foi atualizado.


,osmid,name,qtd_pois,qtd_areas_verdes
0,"[447936235, 447936237]",Freeway,0,0
1,29474897,Freeway,0,0
2,"[27090408, 1187268413]",NaN,0,0
3,"[29210971, 288127021, 29210975, 1282045357]",Freeway,0,1
4,306683729,Rua Albion,0,0
5,451653212,Rua Albion,0,0
6,208803356,Rua Albion,0,2
7,"[451652985, 927886438]",Avenida Ipiranga,0,0
8,"[356396032, 471814874, 927931452, 471814872]",Avenida Joaquim Porto Villanova,0,0
9,927886434,Avenida Ipiranga,0,0


### Cruzamento Espacial com os Dados de Acidentes

Agora vamos cruzar os dados de acidentes (`df_acid_filtered`) com a nossa malha viária (`gdf_ruas_final`) para extrair a variável dependente principal (ou explicativa) para o modelo de ML: **a quantidade de acidentes ocorrida em cada rua**.

In [13]:
import geopandas as gpd
from shapely.geometry import Point

# 1. Converter o DataFrame de acidentes em um GeoDataFrame de pontos
# Nota: criamos a geometria com base em Longitude (X) e Latitude (Y)
gdf_acidentes = gpd.GeoDataFrame(
    df_acid_filtered,
    geometry=gpd.points_from_xy(df_acid_filtered['longitude'], df_acid_filtered['latitude']),
    crs='EPSG:4326'
)

# 2. Projetar os acidentes para coordenadas métricas (mesmo sistema de UTM 22S usado nas ruas)
gdf_acidentes_m = gdf_acidentes.to_crs(epsg=32722)

# 3. Fazer o Join Espacial utilizando o buffer de 50m das ruas
# Isso associa cada acidente à rua correspondente mais próxima dentro de 50 metros
join_ruas_acidentes = gpd.sjoin(gdf_acidentes_m, gdf_ruas_buffer, how='inner', predicate='within')

# 4. Agrupar os acidentes para extrair métricas de interesse para cada segmento de rua (osmid)
# Vamos contar a quantidade de acidentes e somar o total de feridos e mortes
estatisticas_acidentes = join_ruas_acidentes.groupby('osmid').agg(
    qtd_acidentes=('tipo_acid', 'size'),
    total_feridos=('feridos', 'sum'),
    total_mortes=('mortes', 'sum')
).reset_index()

# 5. Mesclar essas novas colunas no nosso DataFrame final de ruas
gdf_final_ml = gdf_ruas_final.merge(estatisticas_acidentes, on='osmid', how='left')

# Preencher com 0 as ruas onde nenhum acidente foi registrado
cols_to_fillna = ['qtd_acidentes', 'total_feridos', 'total_mortes']
gdf_final_ml[cols_to_fillna] = gdf_final_ml[cols_to_fillna].fillna(0).astype(int)

# 6. Salvar o dataset mestre definitivo e pronto para modelagem
gdf_final_ml.to_csv('/content/dataset_final_para_ml.csv', index=False)

print("Mesclagem espacial concluída com sucesso!")
print("O arquivo 'dataset_final_para_ml.csv' está pronto para ser treinado!")
display(gdf_final_ml[['osmid', 'name', 'qtd_pois', 'qtd_areas_verdes', 'qtd_acidentes', 'total_feridos']].head(10))

Mesclagem espacial concluída com sucesso!
O arquivo 'dataset_final_para_ml.csv' está pronto para ser treinado!


,osmid,name,qtd_pois,qtd_areas_verdes,qtd_acidentes,total_feridos
0,"[447936235, 447936237]",Freeway,0,0,1,1
1,29474897,Freeway,0,0,1,4
2,"[27090408, 1187268413]",NaN,0,0,3,1
3,"[29210971, 288127021, 29210975, 1282045357]",Freeway,0,1,17,6
4,306683729,Rua Albion,0,0,99,51
5,451653212,Rua Albion,0,0,26,12
6,208803356,Rua Albion,0,2,102,42
7,"[451652985, 927886438]",Avenida Ipiranga,0,0,51,14
8,"[356396032, 471814874, 927931452, 471814872]",Avenida Joaquim Porto Villanova,0,0,69,23
9,927886434,Avenida Ipiranga,0,0,98,37
